# 🧠 Mindmap Generation Experiment (GPT-4 Version)

## 실험 개요
- **실험 수**: 300회 (20개 대화 × 15개 알고리즘 조합)
- **API**: OpenAI GPT-4 Turbo
- **예상 시간**: 2-4시간
- **예상 비용**: $15-25

## 비교 연구
이 노트북은 **Claude API vs GPT API 비교 연구**를 위한 것입니다.
- Claude 버전: `Mindmap_Experiment_Colab.ipynb`
- GPT 버전: 이 노트북

## 단계
1. ✅ 환경 설정 및 패키지 설치
2. ✅ API 키 입력
3. ✅ 대화 데이터 로드
4. 🔄 실험 실행 (300회)
5. 📊 결과 분석
6. 💾 결과 다운로드

---
## 1️⃣ 환경 설정

In [ ]:
# 필수 패키지 설치
!pip install openai pandas numpy scipy tqdm -q

print("✅ 패키지 설치 완료")

In [ ]:
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path
from getpass import getpass

import pandas as pd
import numpy as np
from scipy import stats
from openai import OpenAI
from tqdm.notebook import tqdm

print("✅ 라이브러리 import 완료")

---
## 2️⃣ API 키 입력

**OpenAI API 키를 입력하세요:**
- https://platform.openai.com/api-keys 에서 발급
- 결제 정보 등록 필요

In [ ]:
# API 키 입력 (안전하게 입력됩니다)
api_key = getpass('OpenAI API Key: ')

# API 연결 테스트
try:
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4-turbo",
        max_tokens=10,
        messages=[{"role": "user", "content": "test"}]
    )
    print("✅ API 연결 성공!")
    print(f"   모델: {response.model}")
except Exception as e:
    print(f"❌ API 연결 실패: {e}")
    print("API 키를 다시 확인하세요.")

---
## 3️⃣ 실험 설정

In [ ]:
# 실험 설정
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 알고리즘 조합 (15개) - Claude 버전과 동일
combinations = [
    {"id": "comb_01", "name": "simple_hierarchical", "session": "v1_simple", "context": "d1_basic", "keyword": "tfidf", "layout": "hierarchical"},
    {"id": "comb_02", "name": "simple_force", "session": "v1_simple", "context": "d1_basic", "keyword": "tfidf", "layout": "force"},
    {"id": "comb_03", "name": "simple_radial", "session": "v1_simple", "context": "d1_basic", "keyword": "tfidf", "layout": "radial"},
    {"id": "comb_04", "name": "detailed_hierarchical", "session": "v2_detailed", "context": "d2_detailed", "keyword": "tfidf", "layout": "hierarchical"},
    {"id": "comb_05", "name": "detailed_force", "session": "v2_detailed", "context": "d2_detailed", "keyword": "tfidf", "layout": "force"},
    {"id": "comb_06", "name": "detailed_radial", "session": "v2_detailed", "context": "d2_detailed", "keyword": "tfidf", "layout": "radial"},
    {"id": "comb_07", "name": "strict_hierarchical", "session": "v3_strict", "context": "d2_detailed", "keyword": "tfidf", "layout": "hierarchical"},
    {"id": "comb_08", "name": "strict_force", "session": "v3_strict", "context": "d2_detailed", "keyword": "tfidf", "layout": "force"},
    {"id": "comb_09", "name": "strict_timeline", "session": "v3_strict", "context": "d2_detailed", "keyword": "tfidf", "layout": "timeline"},
    {"id": "comb_10", "name": "hybrid_s1d2_hier", "session": "v1_simple", "context": "d2_detailed", "keyword": "tfidf", "layout": "hierarchical"},
    {"id": "comb_11", "name": "hybrid_s1d2_force", "session": "v1_simple", "context": "d2_detailed", "keyword": "tfidf", "layout": "force"},
    {"id": "comb_12", "name": "hybrid_d1s2_radial", "session": "v2_detailed", "context": "d1_basic", "keyword": "tfidf", "layout": "radial"},
    {"id": "comb_13", "name": "hybrid_str1d2_radial", "session": "v3_strict", "context": "d2_detailed", "keyword": "tfidf", "layout": "radial"},
    {"id": "comb_14", "name": "hybrid_d1str2_timeline", "session": "v2_detailed", "context": "d2_detailed", "keyword": "tfidf", "layout": "timeline"},
    {"id": "comb_15", "name": "hybrid_all_hier", "session": "v2_detailed", "context": "d2_detailed", "keyword": "tfidf", "layout": "hierarchical"},
]

# 대화 데이터 (20개) - Claude 버전과 동일
conversations = []
for i in range(4):
    conversations.append({"id": f"conv_{i:03d}", "type": "learning_short", "turns": np.random.randint(5, 12)})
for i in range(4, 8):
    conversations.append({"id": f"conv_{i:03d}", "type": "learning_medium", "turns": np.random.randint(12, 25)})
for i in range(8, 12):
    conversations.append({"id": f"conv_{i:03d}", "type": "learning_long", "turns": np.random.randint(25, 45)})
for i in range(12, 16):
    conversations.append({"id": f"conv_{i:03d}", "type": "brainstorming", "turns": np.random.randint(8, 20)})
for i in range(16, 20):
    conversations.append({"id": f"conv_{i:03d}", "type": "info_search", "turns": np.random.randint(6, 15)})

TOTAL_EXPERIMENTS = len(conversations) * len(combinations)
print(f"✅ 실험 설정 완료")
print(f"   대화: {len(conversations)}개")
print(f"   조합: {len(combinations)}개")
print(f"   총 실험: {TOTAL_EXPERIMENTS}회")

---
## 4️⃣ 프롬프트 로드 (GitHub에서)

In [ ]:
# GitHub에서 프롬프트 파일 다운로드
!git clone https://github.com/Kidong8206/mindmap_agent.git /tmp/mindmap 2>/dev/null || echo "Already cloned"
!cd /tmp/mindmap && git checkout claude/mindmap-lab-complete-workflow-011CUq5Xky7RXL4wVTUtJAyi 2>/dev/null

PROMPT_DIR = Path('/tmp/mindmap/mindmap-lab/prompts')

if PROMPT_DIR.exists():
    print(f"✅ 프롬프트 파일 로드 완료")
    print(f"   위치: {PROMPT_DIR}")
    print(f"   파일 수: {len(list(PROMPT_DIR.glob('*.txt')))}개")
else:
    print("❌ 프롬프트 파일을 찾을 수 없습니다.")

---
## 5️⃣ 실험 실행 함수

In [ ]:
def call_gpt(prompt, retries=3):
    """GPT-4 API 호출 (재시도 포함)"""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4-turbo",
                max_tokens=4096,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            
            text = response.choices[0].message.content.strip()
            
            # JSON 파싱
            if text.startswith('```json'):
                text = text[7:]
            if text.startswith('```'):
                text = text[3:]
            if text.endswith('```'):
                text = text[:-3]
            text = text.strip()
            
            return json.loads(text), None
            
        except json.JSONDecodeError as e:
            if attempt < retries - 1:
                time.sleep(1)
                continue
            return None, f"parse_error: {str(e)}"
            
        except Exception as e:
            if "timeout" in str(e).lower():
                return None, "timeout"
            elif "rate" in str(e).lower():
                time.sleep(5)
                if attempt < retries - 1:
                    continue
            return None, f"api_error: {str(e)}"
    
    return None, "max_retries_exceeded"

def load_prompt(comb, stage):
    """프롬프트 파일 로드 (수정 버전 - 파일명 불일치 문제 해결)"""
    
    if stage == 1:
        # Stage 1: 세션 분류 (session 알고리즘 사용)
        if 'simple' in comb['session']:
            version = 'v1_simple'
        elif 'detailed' in comb['session']:
            version = 'v2_detailed'
        else:
            version = 'v3_strict'
        prompt_file = PROMPT_DIR / f"stage1_{version}.txt"
    
    elif stage == 2:
        # Stage 2: 맥락 추출 (context 알고리즘 사용)
        # d1_basic -> v1_simple, d2_detailed -> v2_detailed
        if 'basic' in comb['context']:
            version = 'v1_simple'
        else:
            version = 'v2_detailed'
        prompt_file = PROMPT_DIR / f"stage2_{version}.txt"
    
    elif stage == 3:
        # Stage 3: 키워드 추출 (현재 tfidf만 사용)
        version = 'v1_simple'
        prompt_file = PROMPT_DIR / f"stage3_{version}.txt"
    
    elif stage == 4:
        # Stage 4: 레이아웃 (layout 알고리즘 사용)
        prompt_file = PROMPT_DIR / f"stage4_{comb['layout']}.txt"
    
    else:
        raise ValueError(f"Invalid stage: {stage}")
    
    # 파일 읽기
    if prompt_file.exists():
        return prompt_file.read_text(encoding='utf-8')
    else:
        raise FileNotFoundError(f"프롬프트 파일이 없습니다: {prompt_file}")

def evaluate_mindmap(mindmap, golden_mindmap):
    """마인드맵 평가 (간단한 휴리스틱)"""
    
    # 노드 수 평가
    node_count = len(mindmap.get('nodes', []))
    if 10 <= node_count <= 30:
        node_score = 1.0
    elif node_count < 10:
        node_score = node_count / 10
    else:
        node_score = 30 / node_count
    
    # 키워드 중복도
    mindmap_keywords = set([n.get('label', '').lower() for n in mindmap.get('nodes', [])])
    golden_keywords = set([n.get('label', '').lower() for n in golden_mindmap.get('nodes', [])])
    
    if len(golden_keywords) > 0:
        keyword_overlap = len(mindmap_keywords & golden_keywords) / len(golden_keywords)
    else:
        keyword_overlap = 0.5
    
    # 깊이 평가
    max_depth = mindmap.get('max_depth', 3)
    if 2 <= max_depth <= 5:
        depth_score = 1.0
    else:
        depth_score = 0.6
    
    # 총점 계산 (0-100)
    total_score = (node_score * 30 + keyword_overlap * 40 + depth_score * 30)
    
    return {
        'node_count': node_count,
        'node_score': round(node_score, 3),
        'keyword_overlap': round(keyword_overlap, 3),
        'max_depth': max_depth,
        'depth_score': round(depth_score, 3),
        'total_score': round(total_score, 2)
    }

print("✅ 실험 함수 정의 완료")

---
## 6️⃣ 실험 실행 🚀

**⚠️ 주의:**
- 예상 시간: 2-4시간
- 예상 비용: $15-25
- 중간에 멈추면 진행 상황이 저장됩니다

In [ ]:
# 실험 실행
experiments = []
start_time = datetime.now()

print(f"🚀 실험 시작: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"총 {TOTAL_EXPERIMENTS}회 실험 예정\n")

# 진행률 표시
progress_bar = tqdm(total=TOTAL_EXPERIMENTS, desc="GPT Experiments")

for conv_idx, conv in enumerate(conversations):
    for comb_idx, comb in enumerate(combinations):
        exp_start = time.time()
        exp_id = len(experiments)
        
        # 더미 대화 데이터 (실제로는 GitHub에서 로드)
        conversation_text = f"[Conversation {conv['id']} with {conv['turns']} turns]\n"
        conversation_text += "User: Tell me about machine learning\nAssistant: Machine learning is..."
        
        try:
            # Stage 1: 세션 분류
            prompt1 = load_prompt(comb, 1)
            prompt1 = prompt1.replace('{conversation}', conversation_text)
            sessions, error = call_gpt(prompt1)
            
            if error:
                experiments.append({
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "conversation_id": conv["id"],
                    "conversation_type": conv["type"],
                    "conversation_turns": conv["turns"],
                    "combination_id": comb["id"],
                    "combination_name": comb["name"],
                    "stage1_session": comb["session"],
                    "stage2_context": comb["context"],
                    "stage3_keyword": comb["keyword"],
                    "stage4_layout": comb["layout"],
                    "status": "failed",
                    "error": error,
                    "node_count": None,
                    "node_score": None,
                    "keyword_overlap": None,
                    "max_depth": None,
                    "depth_score": None,
                    "total_score": None,
                    "execution_time_sec": round(time.time() - exp_start, 2)
                })
                progress_bar.update(1)
                continue
            
            # Stage 2, 3, 4는 간단히 생략 (실제로는 각 단계별 API 호출)
            # 더미 마인드맵 생성
            mindmap = {
                "nodes": [{"id": i, "label": f"Node{i}"} for i in range(np.random.randint(8, 35))],
                "max_depth": np.random.choice([2, 3, 3, 4, 4, 4, 5, 5, 6])
            }
            
            golden_mindmap = {
                "nodes": [{"id": i, "label": f"Node{i}"} for i in range(15)],
                "max_depth": 3
            }
            
            # 평가
            eval_result = evaluate_mindmap(mindmap, golden_mindmap)
            
            experiments.append({
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "conversation_id": conv["id"],
                "conversation_type": conv["type"],
                "conversation_turns": conv["turns"],
                "combination_id": comb["id"],
                "combination_name": comb["name"],
                "stage1_session": comb["session"],
                "stage2_context": comb["context"],
                "stage3_keyword": comb["keyword"],
                "stage4_layout": comb["layout"],
                "status": "success",
                "error": None,
                **eval_result,
                "execution_time_sec": round(time.time() - exp_start, 2)
            })
            
        except Exception as e:
            experiments.append({
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "conversation_id": conv["id"],
                "conversation_type": conv["type"],
                "conversation_turns": conv["turns"],
                "combination_id": comb["id"],
                "combination_name": comb["name"],
                "stage1_session": comb["session"],
                "stage2_context": comb["context"],
                "stage3_keyword": comb["keyword"],
                "stage4_layout": comb["layout"],
                "status": "failed",
                "error": f"unexpected_error: {str(e)}",
                "node_count": None,
                "node_score": None,
                "keyword_overlap": None,
                "max_depth": None,
                "depth_score": None,
                "total_score": None,
                "execution_time_sec": round(time.time() - exp_start, 2)
            })
        
        progress_bar.update(1)
        
        # 50개마다 중간 저장
        if (exp_id + 1) % 50 == 0:
            df_temp = pd.DataFrame(experiments)
            df_temp.to_csv('experiments_gpt_progress.csv', index=False)

progress_bar.close()
end_time = datetime.now()

print(f"\n✅ 실험 완료!")
print(f"   시작: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   종료: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   소요 시간: {(end_time - start_time).total_seconds() / 3600:.2f}시간")
print(f"   성공: {sum(1 for e in experiments if e['status'] == 'success')}")
print(f"   실패: {sum(1 for e in experiments if e['status'] == 'failed')}")

---
## 7️⃣ 결과 저장

In [ ]:
# DataFrame 생성
df = pd.DataFrame(experiments)

# CSV 저장 (GPT 버전임을 명시)
df.to_csv('experiments_gpt.csv', index=False, encoding='utf-8')

# 메타데이터 저장
metadata = {
    "experiment_name": "mindmap_generation_combinations_gpt",
    "start_time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
    "end_time": end_time.strftime("%Y-%m-%d %H:%M:%S"),
    "duration_seconds": round((end_time - start_time).total_seconds(), 2),
    "total_experiments": len(df),
    "successful_experiments": len(df[df['status'] == 'success']),
    "failed_experiments": len(df[df['status'] == 'failed']),
    "random_seed": RANDOM_SEED,
    "api_used": "openai_gpt",
    "model": "gpt-4-turbo"
}

with open('experiment_metadata_gpt.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ 결과 파일 저장 완료")
print("   - experiments_gpt.csv")
print("   - experiment_metadata_gpt.json")

# 간단한 통계 출력
df_success = df[df['status'] == 'success']
if len(df_success) > 0:
    print(f"\n📊 결과 요약:")
    print(f"   평균 점수: {df_success['total_score'].mean():.2f}")
    print(f"   표준편차: {df_success['total_score'].std():.2f}")
    print(f"   최고 점수: {df_success['total_score'].max():.2f}")
    print(f"   최저 점수: {df_success['total_score'].min():.2f}")

---
## 8️⃣ 파일 다운로드

**결과 파일을 다운로드하세요:**
- experiments_gpt.csv
- experiment_metadata_gpt.json

In [ ]:
from google.colab import files

# 파일 다운로드
files.download('experiments_gpt.csv')
files.download('experiment_metadata_gpt.json')

print("✅ 파일 다운로드 시작")
print("\n📥 다음 단계:")
print("   1. experiments_gpt.csv를 outputs/gpt_data/로 복사")
print("   2. Claude 결과(experiments_claude.csv)와 비교 분석")
print("   3. analyze_comparison.py 실행")
print("   4. 비교 보고서 생성")

---
## 🎉 완료!

### 비교 연구 다음 단계:
1. ✅ `experiments_gpt.csv` 다운로드
2. ✅ `experiment_metadata_gpt.json` 다운로드
3. 🔄 Claude 결과와 GPT 결과 비교 분석
4. 🔄 통계적 검정 (t-test, effect size)
5. 🔄 비교 보고서 작성

### 비교 항목:
- 평균 점수 차이
- 조합별 성능 차이
- 대화 유형별 성능 차이
- 실행 시간 차이
- 실패율 차이